# 3D Medical Image Preprocessing

A short note for CT images in NIfTI format.

Date: 2026-08-13

Author: Jing

Reference: [3D-Medical-Imaging-Preprocessing-All-you-need](https://github.com/fitushar/3D-Medical-Imaging-Preprocessing-All-you-need)

## 1. Setup

Install the main packages:

```bash
pip install SimpleITK numpy matplotlib
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

## 2. Read and view NIfTI data

SimpleITK uses `(x, y, z)`. The NumPy array uses `(z, y, x)`.

In [ ]:
image_path = Path("data/image.nii.gz")
label_path = Path("data/label.nii.gz")

image_itk = sitk.ReadImage(str(image_path), sitk.sitkFloat32)
label_itk = sitk.ReadImage(str(label_path), sitk.sitkUInt8)

image = sitk.GetArrayFromImage(image_itk)
label = sitk.GetArrayFromImage(label_itk)

print("Array shape:", image.shape)
print("Spacing (x, y, z):", image_itk.GetSpacing())

In [ ]:
z = image.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image[z], cmap="gray")
axes[0].set_title("CT")
axes[1].imshow(label[z], cmap="jet")
axes[1].set_title("Label")
axes[2].imshow(image[z], cmap="gray")
axes[2].imshow(label[z], cmap="jet", alpha=0.4)
axes[2].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 3. Intensity normalization

CT intensity is measured in Hounsfield Units (HU). First clip a useful HU range, then scale it. The best range depends on the task.

In [ ]:
def clip_and_scale_ct(array, low=-1000.0, high=800.0):
    array = np.clip(array, low, high).astype(np.float32)
    return (array - low) / (high - low)


def z_score(array, mask=None):
    values = array[mask > 0] if mask is not None else array
    std = values.std()
    if std == 0:
        return np.zeros_like(array, dtype=np.float32)
    return ((array - values.mean()) / std).astype(np.float32)


image_01 = clip_and_scale_ct(image)

## 4. Resampling

Scans can have different voxel spacing. Use continuous interpolation for an image and nearest-neighbor interpolation for a label.

In [ ]:
def resample(image_itk, out_spacing=(1.0, 1.0, 1.0), is_label=False):
    old_spacing = image_itk.GetSpacing()
    old_size = image_itk.GetSize()
    new_size = [
        int(round(size * spacing / new_spacing))
        for size, spacing, new_spacing in zip(old_size, old_spacing, out_spacing)
    ]

    interpolation = (
        sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear
    )
    default_value = 0.0 if is_label else -1000.0

    return sitk.Resample(
        image_itk,
        new_size,
        sitk.Transform(),
        interpolation,
        image_itk.GetOrigin(),
        out_spacing,
        image_itk.GetDirection(),
        default_value,
        image_itk.GetPixelID(),
    )


image_resampled = resample(image_itk)
label_resampled = resample(label_itk, is_label=True)

## 5. Center crop or pad

This makes all arrays the same shape. A center crop may remove a small target, so check the result.

In [ ]:
def center_crop_or_pad(array, target_shape, pad_value=0):
    slices = []
    padding = []

    for current, target in zip(array.shape, target_shape):
        start = max((current - target) // 2, 0)
        slices.append(slice(start, start + min(current, target)))

        total_pad = max(target - current, 0)
        padding.append((total_pad // 2, total_pad - total_pad // 2))

    cropped = array[tuple(slices)]
    return np.pad(cropped, padding, constant_values=pad_value)


fixed_image = center_crop_or_pad(image, (128, 256, 256), pad_value=-1000)
fixed_label = center_crop_or_pad(label, (128, 256, 256), pad_value=0)

## 6. Histogram matching

Histogram matching changes one scan to look more like a reference scan. It is different from histogram equalization. Use a training image as the reference to avoid data leakage.

In [ ]:
def histogram_match(moving_itk, reference_itk):
    matcher = sitk.HistogramMatchingImageFilter()
    matcher.SetNumberOfHistogramLevels(256)
    matcher.SetNumberOfMatchPoints(20)
    matcher.ThresholdAtMeanIntensityOn()
    return matcher.Execute(moving_itk, reference_itk)

## 7. Maximum intensity projection (MIP)

MIP keeps the maximum value along one axis and creates a 2D image.

In [ ]:
mip = np.max(image, axis=0)
plt.imshow(mip, cmap="gray")
plt.title("Axial MIP")
plt.axis("off");

## 8. Quick checklist

- Check shape, spacing, origin, and direction.
- Keep image and label in the same physical space.
- Use nearest-neighbor interpolation for labels.
- Check labels again after preprocessing.
- Save preprocessing parameters for reproducibility.

## Note

The methods above are useful for learning and small experiments. For deep learning training, libraries such as [MONAI](https://monai.io/) provide tested transforms, data augmentation, caching, and medical image pipelines. TorchIO and nnU-Net are also useful options.

- Apply the same spatial transforms to the image and label.
- Use linear or bilinear interpolation for images.
- Use nearest-neighbor interpolation for labels.
- Choose intensity normalization based on the modality and task.
- Calculate dataset statistics from the training set only.
- Use random augmentation only for training data.
- Check image–label alignment after preprocessing.
- Keep spacing, orientation, and other metadata.
- Use caching or patch-based sampling for large 3D volumes.
- Use a separate deterministic pipeline for validation and test data.

A simple MONAI training pipeline:

```python
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    Orientationd,
    RandFlipd,
    ScaleIntensityRanged,
    Spacingd,
)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(
        keys=["image", "label"],
        axcodes="RAS",
    ),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys="image",
        a_min=-1000,
        a_max=800,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    RandFlipd(
        keys=["image", "label"],
        spatial_axis=0,
        prob=0.5,
    ),
])
```

The preprocessing parameters above are only examples. They should be adjusted for the dataset, imaging modality, target organ, and model.